# BrainWeb · Fisher–KPP tumor growth explorer

Explore a **three-dimensional** tumor-density field on the BrainWeb **T1 ICBM normal 1 mm pn0 rf0** phantom. The matching discrete tissue atlas determines gray matter, white matter and CSF. Start Jupyter from this folder with `uv run jupyter lab tumor_growth_explorer.ipynb`, then **Run All**.

The GUI starts at a 2 mm grid for responsiveness. Set **Δx = 1 mm** for native anatomy. Coordinates are RAS+ in mm: x toward right, y toward anterior, z toward superior. Images use neurological orientation (subject left on image left). The underlying phantom is normal; every tumor here is simulated.

## Model, units and assumptions

$$\frac{\partial n}{\partial t}=\nabla\cdot(D(\mathbf{x})\nabla n)+\rho(\mathbf{x})n\left(1-\frac{n}{K(\mathbf{x})}\right),\quad (D\nabla n)\cdot\nu=0.$$

- $n$: cells/mm³; $D$: mm²/day; $\rho$: day⁻¹; $K$: cells/mm³; time: days.
- The GUI sets $D_{WM}$, $D_{GM}$, $D_{CSF}$ and a smooth spatial multiplier centered on the seed. Class 8 (glial matter) uses GM parameters. All extracerebral tissues are impermeable. CSF never proliferates, and is impermeable when $D_{CSF}=0$.
- Choose a uniform ellipsoid (radii are semiaxes) or a Gaussian (radii are standard deviations, truncated at 3σ); both are restricted to WM/GM/glial tissue initially. The peak density is independent of $K$.
- Fisher–KPP includes simultaneous invasion and proliferation. It does **not** explicitly model a go-or-grow phenotype switch. No treatment, mass effect, necrosis, or DTI fiber anisotropy is included. Parameters are illustrative rather than patient-calibrated.

### Conservative numerical scheme

For a voxel $i$, define a symmetric face coefficient $a_{ij}=D_{ij}/h^2$, with harmonic mean $D_{ij}=2D_iD_j/(D_i+D_j)$. A closed face has zero coefficient. Each internal step first applies

$$n_i^*=n_i^m+\Delta t\sum_{j\sim i}a_{ij}(n_j^m-n_i^m),$$

then the exact local logistic flow

$$n_i^{m+1}=\frac{n_i^*}{e^{-\rho_i\Delta t}+(n_i^*/K_i)(1-e^{-\rho_i\Delta t})}.$$

This is first order in time (Lie splitting and forward Euler diffusion). The conservative centered stencil is second order in smooth regions; tissue interfaces and nearest-neighbor anatomy complicate spatial convergence. The positivity limit used is $\Delta t\le0.9/\max_i\sum_j a_{ij}$. For constant 3D diffusion it is $0.9h^2/(6D)$. There are no periodic boundaries and no density clipping. The requested step is a maximum; smaller steps land exactly on saved frame times. Disable automatic subdivision to reject an unsafe request. Spatially varying $D$ is handled by fluxes, rather than by multiplying a Laplacian by $D$.

The color scale is fixed at $n/K=0\ldots1$ and hides values below 0.02. The 20% and 80% volume thresholds are illustrative; they are not calibrated MRI detection boundaries.

In [ ]:
%matplotlib widget
from IPython.display import display
from brainweb_kpp.data import verify_data
from brainweb_kpp.ui import TumorExplorer

manifest = verify_data()
app = TumorExplorer()
display(app.widget)

## Using the explorer

1. In **Origin & density**, move the coordinate sliders or click an MRI plane. Select tissue preview to check the location. Set three seed radii, its profile and peak density.
2. In **Diffusion & growth**, set tissue diffusion, spatial modulation, proliferation and carrying capacity. Changing K while holding peak density fixed changes the initial fraction n/K.
3. In **Grid & time**, choose Δx, the maximum Δt, total simulated days and output cadence. Press **Run 3D simulation**. First use compiles Numba. Playback and view sliders work after the run; editing simulation parameters requires another run.
4. Use **Export** for the MP4, GIF and supporting results. Rendering is synchronous and can take longer than solving.

The app stores 3D frames by default so you can change slice positions during playback. A native 1 mm float32 volume is about 28 MB; 61 frames use about 1.7 GB plus solver and rendering arrays. At 0.5 mm, costs rise sharply. Uncheck 3D storage to keep only three planes through the origin (used by the CLI). Export always shows planes through the original seed, even if you explored other slices.

Do a refinement study (smaller Δt and Δx) before interpreting quantitative differences. Resampling at 0.5 mm does not add anatomical information.

## Programmatic maps and batch runs

The same solver accepts custom arrays for D(x), ρ(x), and K(x), on the selected `brain` grid. Arrays must use the same canonical RAS axes. The notebook widgets use uniform tissue proliferation and uniform K. Custom spatial K can be exceeded locally by incoming diffusion; values are not clipped. Use the Python result arrays for such experiments: the standard media renderer assumes the scalar K in Parameters for its color scale.

The following cell is optional; it is left unexecuted to avoid starting a second simulation.

In [2]:
# from dataclasses import replace
# from brainweb_kpp import Parameters, load_brainweb, simulate
# from brainweb_kpp.model import fields
# from brainweb_kpp.render import save_result, export_media
# brain = load_brainweb(spacing=1.0)
# p = replace(Parameters(), frame_days=1.5)
# D, rho, K = fields(brain, p)
# # Modify coefficient arrays here if needed.
# result = simulate(brain, p, diffusion=D, proliferation=rho, capacity=K)
# save_result(result, "outputs/custom")
# export_media(result, "outputs/custom", fps=12)

## Sources and reproducibility

- [BrainWeb normal anatomical model and tissue classes](https://brainweb.bic.mni.mcgill.ca/anatomic_normal.html).
- [BrainWeb simulated database and citation information](https://brainweb.bic.mni.mcgill.ca/brainweb/).
- Collins et al. (1998), *Design and construction of a realistic digital brain phantom*, IEEE TMI 17(3), 463–468. [DOI](https://doi.org/10.1109/42.712135).
- Everett et al. (2020), *A tutorial review of mathematical techniques for quantifying tumor heterogeneity*. [DOI](https://doi.org/10.3934/mbe.2020207), as referenced in the supplied slides.

The bundled manifest records data checksums. `uv.lock` pins the environment. Each export records parameters, grid, actual numerical step, data provenance, library versions and metrics. See README.md for the independent animation command and numerical validation.